# ***Train-Test Split***

This notebook performs a recording-level stratified train-test split.

Why?

- Prevents data leakage due to windowing
- Prevents data leakage due to augmentation
- Preserves class distribution
- Keeps all windows and augmentations of a recording together


"The dataset was split at the recording level rather than at the window level. A stratified 80/20 split was performed on the unique recording IDs to preserve class proportions. After determining the training and testing recordings, all feature vectors (including all windows and augmented samples) belonging to each recording were assigned to the corresponding split. This approach prevents data leakage between training and testing datasets."

In [8]:
!pip install -q scikit-learn

In [9]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split

In [10]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
PROJECT_DIR = Path("/content/drive/MyDrive/Underwater Audio Data")

FEATURES_PATH = PROJECT_DIR /"features" / "audio_features.csv"

SPLIT_DIR = PROJECT_DIR / "splits"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

In [12]:
df = pd.read_csv(FEATURES_PATH)

print(df.shape)

df.head()

(7897, 158)


,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,...,chroma_8,chroma_9,chroma_10,chroma_11,chroma_12,spectral_centroid,zero_crossing_rate,recording_id,filename,label
0,-162.371080,114.084236,17.442232,34.860504,11.792038,8.038345,2.341038,10.792850,-1.959563,5.029838,...,0.425020,0.518890,0.663195,0.721205,0.781813,1065.627842,0.036377,Tanker_8,Tanker_8_chunk23.wav,Vessels
1,9.819119,93.509330,-5.404236,16.587270,-1.691457,17.680866,5.285886,7.211845,-2.952189,3.849669,...,0.572907,0.546290,0.639305,0.737739,0.709395,1673.987016,0.091672,Tanker_9,Tanker_9_chunk14.wav,Vessels
2,-108.764560,124.395030,21.671942,28.292772,15.838627,16.726170,9.449649,11.452087,4.408017,9.027928,...,0.363379,0.445676,0.613555,0.679194,0.779457,815.989303,0.021282,Tanker_8,Tanker_8_chunk44.wav,Vessels
3,6.157171,95.591300,-4.524734,16.876825,-1.962507,17.545330,3.787380,5.727674,-3.289775,3.570226,...,0.543499,0.504719,0.572385,0.700327,0.650260,1663.169850,0.096285,Tanker_9,Tanker_9_chunk15.wav,Vessels
4,5.273881,96.318910,-3.680939,16.376200,-0.308045,17.221512,3.711733,5.968836,-3.987965,3.100595,...,0.506153,0.472855,0.536263,0.640560,0.590224,1640.076919,0.093610,Tanker_9,Tanker_9_chunk16.wav,Vessels


Verify coloums

In [13]:
print(df.columns.tolist())

['mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12', 'mfcc_13', 'mel_1', 'mel_2', 'mel_3', 'mel_4', 'mel_5', 'mel_6', 'mel_7', 'mel_8', 'mel_9', 'mel_10', 'mel_11', 'mel_12', 'mel_13', 'mel_14', 'mel_15', 'mel_16', 'mel_17', 'mel_18', 'mel_19', 'mel_20', 'mel_21', 'mel_22', 'mel_23', 'mel_24', 'mel_25', 'mel_26', 'mel_27', 'mel_28', 'mel_29', 'mel_30', 'mel_31', 'mel_32', 'mel_33', 'mel_34', 'mel_35', 'mel_36', 'mel_37', 'mel_38', 'mel_39', 'mel_40', 'mel_41', 'mel_42', 'mel_43', 'mel_44', 'mel_45', 'mel_46', 'mel_47', 'mel_48', 'mel_49', 'mel_50', 'mel_51', 'mel_52', 'mel_53', 'mel_54', 'mel_55', 'mel_56', 'mel_57', 'mel_58', 'mel_59', 'mel_60', 'mel_61', 'mel_62', 'mel_63', 'mel_64', 'mel_65', 'mel_66', 'mel_67', 'mel_68', 'mel_69', 'mel_70', 'mel_71', 'mel_72', 'mel_73', 'mel_74', 'mel_75', 'mel_76', 'mel_77', 'mel_78', 'mel_79', 'mel_80', 'mel_81', 'mel_82', 'mel_83', 'mel_84', 'mel_85', 'mel_86', 'mel_87', 'mel

Check for missing values.


In [14]:
print(df.isnull().sum())

mfcc_1                0
mfcc_2                0
mfcc_3                0
mfcc_4                0
mfcc_5                0
                     ..
spectral_centroid     0
zero_crossing_rate    0
recording_id          0
filename              0
label                 0
Length: 158, dtype: int64


Original Sample Distribution:

In [15]:
print(df["label"].value_counts())

label
Vessels       4105
Biological    2160
Ambience      1632
Name: count, dtype: int64


Recording Distribution:

In [16]:
print(df.groupby("label")["recording_id"].nunique())

label
Ambience       18
Biological    109
Vessels        85
Name: recording_id, dtype: int64


Creating Recirding level Dataframe



In [17]:
recordings = (
    df[["recording_id", "label"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(recordings.head())

  recording_id    label
0     Tanker_8  Vessels
1     Tanker_9  Vessels
2    Tanker_10  Vessels
3    Tanker_11  Vessels
4    Tanker_12  Vessels


Verify Number of Recordings

In [19]:
print("Unique Recordings:", len(recordings))

print()

print(recordings["label"].value_counts())

Unique Recordings: 212

label
Biological    109
Vessels        85
Ambience       18
Name: count, dtype: int64


Stratified split recording level


In [20]:
train_recordings, test_recordings = train_test_split(
    recordings,
    test_size=0.20,
    stratify=recordings["label"],
    random_state=42
)

verify recording split



In [21]:
print("Training Recordings")

print(train_recordings["label"].value_counts())

print()

print("Testing Recordings")

print(test_recordings["label"].value_counts())

Training Recordings
label
Biological    87
Vessels       68
Ambience      14
Name: count, dtype: int64

Testing Recordings
label
Biological    22
Vessels       17
Ambience       4
Name: count, dtype: int64


get recording ids

In [23]:
train_ids = train_recordings["recording_id"]

test_ids = test_recordings["recording_id"]

build final train/test dataframes

In [24]:
train_df = df[
    df["recording_id"].isin(train_ids)
].reset_index(drop=True)

test_df = df[
    df["recording_id"].isin(test_ids)
].reset_index(drop=True)

Verify no leakage

In [25]:
overlap = (
    set(train_df["recording_id"])
    &
    set(test_df["recording_id"])
)

print("Overlapping Recordings:", len(overlap))

Overlapping Recordings: 0


Verify final dataset

In [26]:
print("Training Samples")

print(train_df["label"].value_counts())

print()

print("Testing Samples")

print(test_df["label"].value_counts())

Training Samples
label
Vessels       3154
Biological    1660
Ambience      1272
Name: count, dtype: int64

Testing Samples
label
Vessels       951
Biological    500
Ambience      360
Name: count, dtype: int64


Separate features and labels

In [27]:
X_train = train_df.drop(
    columns=[
        "label",
        "filename",
        "recording_id"
    ]
)

y_train = train_df["label"]

X_test = test_df.drop(
    columns=[
        "label",
        "filename",
        "recording_id"
    ]
)

y_test = test_df["label"]

Verify shapes

In [28]:
print("X_train:", X_train.shape)

print("X_test :", X_test.shape)

print()

print("y_train:", y_train.shape)

print("y_test :", y_test.shape)

X_train: (6086, 155)
X_test : (1811, 155)

y_train: (6086,)
y_test : (1811,)


Save Split

In [29]:
X_train.to_csv(
    SPLIT_DIR / "X_train.csv",
    index=False
)

X_test.to_csv(
    SPLIT_DIR / "X_test.csv",
    index=False
)

y_train.to_csv(
    SPLIT_DIR / "y_train.csv",
    index=False
)

y_test.to_csv(
    SPLIT_DIR / "y_test.csv",
    index=False
)

print("Train/Test split saved successfully.")

Train/Test split saved successfully.
